# Gypsum Ceilings Data Analysis and Model Preparation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv("ceilings_gypsum_final_reclassified.csv")
df.head()

## 3. Initial Inspection

In [ ]:
df.info()
df.describe(include="all")
df.nunique()

## 4. Data Quality Checks

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df[df["Price_EGP"] <= 0]

## 5. Data Cleaning

In [ ]:
df_clean = df.copy()

text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["Price_EGP"]).reset_index(drop=True)

df_clean.head()

## 6. Quality Level Analysis

In [ ]:
df_clean.groupby("Quality_Level")["Price_EGP"].describe()

In [ ]:
quality_price = (
    df_clean.groupby("Quality_Level", as_index=False)["Price_EGP"]
    .mean()
)

quality_price["Quality_Level"] = pd.Categorical(
    quality_price["Quality_Level"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

quality_price = quality_price.sort_values("Quality_Level")

sns.barplot(data=quality_price, x="Quality_Level", y="Price_EGP")
plt.title("Average Price by Quality Level")
plt.show()

## 7. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean["Price_EGP"], bins=30, kde=True)
plt.title("Price Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_clean,
    x="Quality_Level",
    y="Price_EGP",
    order=["Low", "Medium", "High"]
)
plt.title("Price Distribution by Quality Level")
plt.show()

In [ ]:
if "Ceiling_Type" in df_clean.columns:
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=df_clean,
        x="Ceiling_Type",
        y="Price_EGP",
        estimator="mean"
    )
    plt.xticks(rotation=45)
    plt.title("Average Price by Ceiling_Type")
    plt.show()

## 8. Outlier Analysis

In [ ]:
Q1 = df_clean["Price_EGP"].quantile(0.25)
Q3 = df_clean["Price_EGP"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean["Price_EGP"] < lower_bound) |
    (df_clean["Price_EGP"] > upper_bound)
]

outliers.head()

## 9. Feature Engineering

In [ ]:
df_model = df_clean.copy()

df_model["Apartment_Area_m2"] = 100

df_model["Estimated_Quantity"] = np.ceil(
    df_model["Apartment_Area_m2"]
    / df_model["Coverage_m2"]
)

df_model["Estimated_Total_Cost"] = (
    df_model["Estimated_Quantity"]
    * df_model["Price_EGP"]
)

df_model.head()

## 10. Apartment Requirement Logic

In [ ]:
def calculate_ceiling_requirements(
    apartment_area,
    rooms,
    coverage_ratio=1
):
    ceiling_area = apartment_area * coverage_ratio

    return {
        "Apartment_Area_m2": apartment_area,
        "Rooms": rooms,
        "Estimated_Ceiling_Area_m2": ceiling_area
    }

## 11. Cost Estimation

In [ ]:
def estimate_category_cost(data, required_quantity, quality_level=None):
    filtered = data.copy()

    if quality_level is not None:
        filtered = filtered[
            filtered["Quality_Level"] == quality_level
        ]

    average_price = filtered["Price_EGP"].mean()

    return required_quantity * average_price

## 12. Budget-Based Recommendation

In [ ]:
def recommend_products(data, budget, quality_level, product_type=None):
    filtered = data[
        data["Quality_Level"] == quality_level
    ].copy()

    if product_type is not None:
        matching_columns = [
            col for col in filtered.columns
            if "Type" in col or "Category" in col or "Material" in col
        ]

        masks = []

        for col in matching_columns:
            masks.append(
                filtered[col].astype(str).str.contains(
                    str(product_type),
                    case=False,
                    na=False
                )
            )

        if masks:
            combined_mask = masks[0]

            for mask in masks[1:]:
                combined_mask = combined_mask | mask

            filtered = filtered[combined_mask]

    filtered = filtered[
        filtered["Price_EGP"] <= budget
    ]

    return filtered.sort_values("Price_EGP")

## 13. Multi-File Model Integration Structure

In [ ]:
df_model["Finishing_Category"] = "Gypsum Ceilings"

common_columns = [
    "Finishing_Category",
    "Category",
    "Subcategory",
    "Product_Name",
    "Brand",
    "Quality_Level",
    "Price_EGP",
    "Unit",
    "Quantity_Rule",
    "Rule_Value",
    "Required_For",
    "Optional"
]

category_for_master_model = df_model[
    [col for col in common_columns if col in df_model.columns]
].copy()

category_for_master_model.head()

## 14. Prepare Data for Future Master Model

In [ ]:
category_model_data = category_for_master_model.copy()

category_model_data = category_model_data.dropna(
    subset=["Price_EGP"]
).reset_index(drop=True)

category_model_data.info()

## 15. Baseline Price Prediction Model

In [ ]:
df_ml = df_model.copy()

candidate_features = [
    col for col in df_ml.columns
    if col not in [
        "Price_EGP",
        "Quality_Level",
        "Quality_Price_Band",
        "Estimated_Total_Cost"
    ]
]

X = df_ml[candidate_features]
y = df_ml["Price_EGP"]

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

## 16. Train and Evaluate Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

baseline_model.fit(X_train, y_train)

predictions = baseline_model.predict(X_test)

model_results = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
    "R2": r2_score(y_test, predictions)
}

model_results

## 17. Actual vs Predicted

In [ ]:
comparison = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": predictions
})

comparison.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=comparison,
    x="Actual_Price",
    y="Predicted_Price"
)
plt.title("Actual vs Predicted Price")
plt.show()

## 18. Final Validation and Export

In [ ]:
model_ready_data = df_model.copy()

model_ready_data.to_csv(
    "ceilings_gypsum_model_ready_reclassified.csv",
    index=False
)

model_ready_data.head()